# Diputrax

Analisis de patrones de reclutamiento para comites legislativos de la camra de diputados del Congreso de la Union de los Estados Unidos Mexicanos.

# 1. Resumen ejecutivo

resumen

## 1.1 Contexto y definicion del area de interes

Contexto


Area de interes
Patrones de reclutamiento de comisiones

## 1.2 Objetivos y criterios de exito

Objetivo primario
d

Objetivos secundarios
d

Directivas de la evaluacion de modelos
d

## 1.3 Datos

kjkjkkk

## 1.4 Criterios de interpretabilidad del modelo


Se considera que 

## 1.5 Alcance del proyecto

dfdfdf

## 1.6 Fuera de alcance del proyecto

sadadasdasdasd

# 2. Estrategia metodológica

In [ ]:
# =========================
# Importación de dependencias y carga de datos
# =========================

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display, HTML

sns.set_theme(style="whitegrid", context="talk", palette="deep")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["legend.frameon"] = False
pd.set_option("display.max_columns", 100)

DATA_PATH = Path("data/database/clean/diputados_20260421_205712.parquet")
REPORT_PATH = Path("reports/eda/eda_report_205712.md")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No existe archivo parquet: {DATA_PATH}")
if not REPORT_PATH.exists():
    raise FileNotFoundError(f"No existe reporte base: {REPORT_PATH}")

df = pd.read_parquet(DATA_PATH).copy()
print(f"Fuente de datos: {DATA_PATH}")
print(f"Reporte base: {REPORT_PATH}")
print(f"Dimensión: {df.shape[0]:,} filas x {df.shape[1]:,} columnas")

roman_map = {
    57: "LVII", 58: "LVIII", 59: "LIX", 60: "LX", 61: "LXI",
    62: "LXII", 63: "LXIII", 64: "LXIV", 65: "LXV", 66: "LXVI",
}

era_map = {
    57: "ERA_1", 58: "ERA_1", 59: "ERA_1",
    60: "ERA_2", 61: "ERA_2", 62: "ERA_2",
    63: "ERA_3", 64: "ERA_3", 65: "ERA_3",
    66: "ERA_4",
}

era_label_map = {
    "ERA_1": "ERA_1 - LVII-LIX",
    "ERA_2": "ERA_2 - LX-LXII",
    "ERA_3": "ERA_3 - LXIII-LXV",
    "ERA_4": "ERA_4 - LXVI",
}

era_nombre_map = {
    "ERA_1 - LVII-LIX": "Primera época",
    "ERA_2 - LX-LXII": "Segunda época",
    "ERA_3 - LXIII-LXV": "Tercera época",
    "ERA_4 - LXVI": "Cuarta época",
}

era_order = ["ERA_1", "ERA_2", "ERA_3", "ERA_4"]
era_nombre_order = ["Primera época", "Segunda época", "Tercera época", "Cuarta época"]

plot_df = df.copy()
plot_df["legislatura_roman"] = plot_df["legislatura_num"].map(roman_map)
plot_df["era"] = plot_df["legislatura_num"].map(era_map)
plot_df["era_label"] = plot_df["era"].map(era_label_map).map(era_nombre_map)

if plot_df["era"].isna().any():
    missing_legs = sorted(plot_df.loc[plot_df["era"].isna(), "legislatura_num"].dropna().unique().tolist())
    raise ValueError(f"Legislaturas sin era asignada: {missing_legs}")

plot_df["era"] = pd.Categorical(plot_df["era"], categories=era_order, ordered=True)
plot_df["era_label"] = pd.Categorical(plot_df["era_label"], categories=era_nombre_order, ordered=True)

plot_df[["legislatura_num", "legislatura_roman", "era", "era_label"]].drop_duplicates().sort_values("legislatura_num")

resumen_era = (
    plot_df.groupby("era_label")
    .agg(
        filas=("diputado_id", "size"),
        legislaturas=("legislatura_num", lambda s: ", ".join(sorted({roman_map[x] for x in s}))),
        diputados_unicos=("diputado_id", "nunique"),
        edad_promedio=("edad_al_tomar_cargo", "mean"),
        comisiones_promedio=("n_comisiones", "mean"),
        nodales_promedio=("n_comisiones_nodales", "mean"),
        tematicas_promedio=("n_comisiones_tematicas", "mean"),
        lastre_promedio=("n_comisiones_lastre", "mean"),
    )
    .reset_index()
)

display(resumen_era.round(2))

dfdfd

dfdf


In [ ]:
# ============================================================
# METADATOS DE COLUMNAS
# ============================================================
# Este diccionario define, para cada columna del DataFrame,
# dos cosas:
# 1) el grupo temático al que pertenece
# 2) una descripción breve de su significado
#
# La estructura es:
# "nombre_columna": ("Grupo temático", "Descripción")
COLUMN_META = {
    # ── Identificadores ──────────────────────────────────────
    "diputado_id":               ("Identificadores", "Hash único del diputado"),
    "referencia":                ("Identificadores", "ID numérico en la fuente original"),
    "nombre":                    ("Identificadores", "Nombre completo del diputado"),
    "legislatura_nombre":        ("Identificadores", "Nombre romano de la legislatura (ej. LIX)"),
    "legislatura_num":           ("Identificadores", "Número ordinal de la legislatura"),
    "partido_nombre":            ("Identificadores", "Nombre completo del partido político"),
    "partido":                   ("Identificadores", "Siglas del partido"),
    "partido_mayoria":           ("Identificadores", "Partido con mayoría en esa legislatura"),
    "es_partido_mayoria":        ("Identificadores", "1 si el diputado pertenece al partido mayoritario"),
    "source_file":               ("Identificadores", "Archivo CSV de origen"),

    # ── Datos personales ─────────────────────────────────────
    "y_nacimiento":              ("Datos personales", "Año de nacimiento"),
    "edad_al_tomar_cargo":       ("Datos personales", "Edad al inicio del mandato"),
    "grado_estudios_ord":        ("Datos personales", "Nivel educativo ordinal (0–7)"),
    "area_formacion":            ("Datos personales", "Área disciplinaria de estudios"),
    "en_licencia":               ("Datos personales", "True si estuvo en licencia durante el periodo"),
    "suplente_referencia":       ("Datos personales", "ID de referencia del suplente"),
    "tiene_suplente":            ("Datos personales", "1 si tiene suplente registrado"),
    "mayoria_relativa":          ("Datos personales", "1 si ganó por mayoría relativa (vs representación proporcional)"),
    "entidad_codigo":            ("Datos personales", "Código de 3 letras del estado"),
    "distrito_circ":             ("Datos personales", "Número y nombre del distrito o circunscripción"),

    # ── Comisiones ───────────────────────────────────────────
    "n_comisiones":              ("Comisiones", "Total de comisiones ordinarias"),
    "n_comisiones_especiales":   ("Comisiones", "Total de comisiones especiales"),
    "n_presidencias":            ("Comisiones", "Número de presidencias de comisión"),
    "n_secretarias":             ("Comisiones", "Número de secretarías de comisión"),
    "presidente_comision":       ("Comisiones", "1 si presidió al menos una comisión"),
    "lider_comision":            ("Comisiones", "1 si fue presidente o secretario de alguna comisión"),
    "n_comisiones_nodales":      ("Comisiones", "Comisiones de alta influencia legislativa"),
    "n_comisiones_tematicas":    ("Comisiones", "Comisiones de política sectorial"),
    "n_comisiones_lastre":       ("Comisiones", "Comisiones de bajo perfil político"),

    # ── Trayectoria legislativa ──────────────────────────────
    "fue_diputado_local":        ("Trayectoria legislativa", "1 si fue diputado local antes"),
    "fue_diputado_federal":      ("Trayectoria legislativa", "1 si fue diputado federal en legislatura previa"),
    "fue_senador":               ("Trayectoria legislativa", "1 si fue senador"),
    "n_cargos_legislativos_prev":("Trayectoria legislativa", "Suma de cargos legislativos previos"),
    "n_trayectoria_legislativa": ("Trayectoria legislativa", "Conteo ponderado de experiencia legislativa"),

    # ── Trayectoria administrativa ───────────────────────────
    "n_trayectoria_admin":       ("Trayectoria administrativa", "Conteo total de cargos administrativos"),
    "fue_presidente_mun":        ("Trayectoria administrativa", "1 si fue presidente municipal"),
    "fue_presidente_org":        ("Trayectoria administrativa", "1 si presidió algún organismo"),
    "fue_director_general":      ("Trayectoria administrativa", "1 si fue director general"),
    "fue_secretario_cargo":      ("Trayectoria administrativa", "1 si fue secretario de despacho"),
    "fue_subsecretario":         ("Trayectoria administrativa", "1 si fue subsecretario"),
    "fue_director":              ("Trayectoria administrativa", "1 si fue director (área)"),
    "fue_coordinador":           ("Trayectoria administrativa", "1 si fue coordinador"),
    "fue_delegado":              ("Trayectoria administrativa", "1 si fue delegado"),
    "fue_asesor":                ("Trayectoria administrativa", "1 si fue asesor"),
    "fue_regidor":               ("Trayectoria administrativa", "1 si fue regidor"),
    "fue_sindico":               ("Trayectoria administrativa", "1 si fue síndico"),
    "admin_en_partido":          ("Trayectoria administrativa", "1 si tuvo cargo administrativo en partido"),
    "admin_en_sindicato":        ("Trayectoria administrativa", "1 si tuvo cargo en sindicato"),
    "admin_en_universidad":      ("Trayectoria administrativa", "1 si tuvo cargo en universidad"),
    "admin_en_gobierno_fed":     ("Trayectoria administrativa", "1 si tuvo cargo en gobierno federal"),
    "admin_en_gobierno_est":     ("Trayectoria administrativa", "1 si tuvo cargo en gobierno estatal"),
    "admin_en_gobierno_mun":     ("Trayectoria administrativa", "1 si tuvo cargo en gobierno municipal"),
    "nivel_cargo_max":           ("Trayectoria administrativa", "Nivel máximo de cargo administrativo (0–5)"),

    # ── Trayectoria juvenil ──────────────────────────────────
    "tiene_exp_juvenil":         ("Trayectoria juvenil", "1 si tiene experiencia en organizaciones juveniles"),
    "lider_juvenil_partido":     ("Trayectoria juvenil", "1 si fue líder juvenil de partido"),
    "lider_juvenil_gobierno":    ("Trayectoria juvenil", "1 si tuvo cargo juvenil en gobierno"),
    "miembro_org_juvenil":       ("Trayectoria juvenil", "1 si fue miembro de organización juvenil"),
    "nivel_liderazgo_juvenil":   ("Trayectoria juvenil", "Nivel de liderazgo juvenil (0–3)"),

    # ── Formación académica ──────────────────────────────────
    "tiene_posgrado":            ("Formación académica", "1 si tiene posgrado"),
    "tiene_doctorado":           ("Formación académica", "1 si tiene doctorado"),
    "estudios_en_extranjero":    ("Formación académica", "1 si estudió en el extranjero"),
    "univ_publica":              ("Formación académica", "1 si estudió en universidad pública"),
    "univ_privada":              ("Formación académica", "1 si estudió en universidad privada"),
    "univ_extranjera":           ("Formación académica", "1 si estudió en universidad extranjera"),
    "acad_unam":                 ("Formación académica", "1 si estudió en la UNAM"),
    "acad_itesm":                ("Formación académica", "1 si estudió en el ITESM (Tec de Monterrey)"),
    "acad_itam":                 ("Formación académica", "1 si estudió en el ITAM"),
    "acad_ibero":                ("Formación académica", "1 si estudió en la Iberoamericana"),
    "acad_udg":                  ("Formación académica", "1 si estudió en la UdG"),
    "acad_ipn":                  ("Formación académica", "1 si estudió en el IPN"),
    "acad_uam":                  ("Formación académica", "1 si estudió en la UAM"),
    "acad_anahuac":              ("Formación académica", "1 si estudió en la Anáhuac"),
    "acad_uanl":                 ("Formación académica", "1 si estudió en la UANL"),
    "acad_uv":                   ("Formación académica", "1 si estudió en la UV"),

    # ── Conteos de trayectoria ───────────────────────────────
    "n_trayectoria_politica":    ("Conteos de trayectoria", "Conteo de cargos políticos en general"),
    "n_trayectoria_empresarial": ("Conteos de trayectoria", "Conteo de cargos en sector privado"),
    "n_investigacion_docencia":  ("Conteos de trayectoria", "Conteo de actividades académicas/investigación"),
    "n_organos_gobierno":        ("Conteos de trayectoria", "Conteo de participaciones en órganos de gobierno"),
}

# ============================================================
# CONSTRUCCIÓN DE FILAS DEL RESUMEN DE ESQUEMA
# ============================================================
# Aquí se va a construir una lista de diccionarios, donde cada
# diccionario representa una fila del reporte de esquema.
rows = []

# Recorremos todas las columnas del DataFrame original.
for col in df.columns:
    # Obtiene el tipo de dato de la columna como texto.
    # Ejemplo: int64, float64, object, bool, category, etc.
    dtype = str(df[col].dtype)

    # Cuenta cuántos valores nulos hay en esa columna.
    nulls = df[col].isnull().sum()

    # Calcula el porcentaje de nulos respecto del total de filas.
    pct_null = f"{nulls / len(df) * 100:.1f}%"

    # Cuenta cuántos valores únicos tiene la columna.
    # Por defecto, nunique() no cuenta NaN.
    nuniq = df[col].nunique()

    # Toma un ejemplo de valor no nulo de la columna.
    # Si toda la columna es nula, deja cadena vacía.
    sample = df[col].dropna().iloc[0] if nulls < len(df) else ""

    # Busca metadatos de la columna en COLUMN_META.
    # Si no encuentra la columna, asigna grupo "Otros"
    # y descripción vacía.
    grupo, desc = COLUMN_META.get(col, ("Otros", ""))

    # Agrega una fila resumen a la lista rows.
    rows.append({
        "Grupo": grupo,               # categoría temática de la variable
        "Columna": col,               # nombre de la columna
        "Tipo": dtype,                # tipo de dato
        "Únicos": nuniq,              # cantidad de valores distintos
        "Nulos": nulls,               # cantidad de valores nulos
        "% Nulo": pct_null,           # porcentaje de valores nulos
        "Ejemplo": str(sample),       # ejemplo de un valor observado
        "Descripción": desc,          # descripción semántica de la variable
    })

# Convierte la lista de filas en un DataFrame.
# Este DataFrame será el esquema descriptivo final.
schema = pd.DataFrame(rows)

# ============================================================
# PALETA DE COLORES MEJORADA PARA LEGIBILIDAD
# ============================================================
# Se usan colores muy claros pero con mejor contraste visual
# entre categorías. La idea es:
# - mantener texto oscuro legible
# - diferenciar grupos sin saturar la vista
# - evitar tonos demasiado pálidos que casi no se distingan
GROUP_COLORS = {
    "Identificadores":           "#E8F0FE",  # azul claro
    "Datos personales":          "#FFF4CC",  # amarillo suave
    "Comisiones":                "#EAF7EA",  # verde suave
    "Trayectoria legislativa":   "#FCE8F3",  # rosa suave
    "Trayectoria administrativa":"#EFE7FB",  # violeta suave
    "Trayectoria juvenil":       "#FFF0E1",  # naranja suave
    "Formación académica":       "#E6F7FA",  # turquesa suave
    "Conteos de trayectoria":    "#EEF2F7",  # gris azulado suave
    "Otros":                     "#FFFFFF",  # blanco
}

# ============================================================
# FUNCIÓN DE ESTILO POR FILA
# ============================================================
# Esta función recibe una fila del DataFrame schema
# y devuelve una lista de estilos CSS, uno por celda de la fila.
def row_color(row):
    # Busca el color según el grupo temático de la fila.
    color = GROUP_COLORS.get(row["Grupo"], "#FFFFFF")

    # Devuelve el estilo CSS para cada celda de esa fila.
    # Además del fondo, fijamos color de texto oscuro para asegurar contraste.
    return [f"background-color: {color}; color: #1F2937;" for _ in row]

# ============================================================
# CONSTRUCCIÓN DE LA TABLA ESTILIZADA
# ============================================================
styled = (
    schema.style

    # Aplica la función row_color fila por fila.
    .apply(row_color, axis=1)

    # Define propiedades generales para todas las celdas.
    .set_properties(**{
        "text-align": "left",
        "font-size": "13px",
        "white-space": "normal",   # permite salto de línea
    })

    # Define estilos CSS más específicos para encabezados y celdas.
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#0F172A"),   # encabezado oscuro
                ("color", "#FFFFFF"),              # texto blanco
                ("font-size", "13px"),
                ("text-align", "left"),
                ("padding", "8px 10px"),
                ("border-bottom", "2px solid #334155"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("padding", "6px 10px"),
                ("border-bottom", "1px solid #D9E2EC"),
                ("vertical-align", "top"),
            ],
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("font-family", "Arial, sans-serif"),
            ],
        },
    ])

    # Oculta el índice del DataFrame en la tabla renderizada.
    .hide(axis="index")
)



In [ ]:
# ============================================================
# RESUMEN AGRUPADO DEL ESQUEMA
# ============================================================
# Este bloque toma el DataFrame "schema" y construye una tabla
# resumen por grupo temático de variables.

summary = (
    # Agrupa las filas de "schema" por la columna "Grupo".
    # sort=False conserva el orden original de aparición de los grupos
    # en lugar de ordenarlos alfabéticamente.
    schema.groupby("Grupo", sort=False)

    # Para cada grupo calcula agregaciones:
    # - n_columnas: cuenta cuántas columnas pertenecen a ese grupo
    # - total_nulos: suma la cantidad total de valores nulos de todas
    #   las variables dentro de ese grupo
    .agg(
        n_columnas=("Columna", "count"),
        total_nulos=("Nulos", "sum")
    )

    # Convierte el índice del groupby nuevamente en una columna normal.
    # Esto facilita mostrar el resultado como tabla estándar.
    .reset_index()
)

# ============================================================
# VISUALIZACIÓN ESTILIZADA DEL RESUMEN
# ============================================================
# display(...) muestra el resultado en un notebook de Jupyter
# con formato enriquecido en vez de texto plano.

display(
    summary.style

    # Aplica propiedades generales a todas las celdas:
    # - alineación a la izquierda
    # - tamaño de fuente de 13 px
    .set_properties(**{
        "text-align": "left",
        "font-size": "13px"
    })

    # Define estilos CSS para los encabezados de la tabla.
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#1e293b"),  # fondo oscuro del encabezado
                ("color", "white"),               # texto blanco
                ("padding", "6px 10px"),         # espacio interno
                ("text-align", "left")           # alinear encabezados a la izquierda
            ]
        }
    ])

    # Oculta la columna del índice para que no se muestre en la tabla final.
    .hide(axis="index")
)

## 2.1 Análisis de los datos y evaluación de su calidad

### Datos disponibles (legisdatamxsil)

Los datos provienen del Sistema de Información Legislativa (SIL) de la Secretaría de Gobernación de México ([sil.gobernacion.gob.mx](https://sil.gobernacion.gob.mx)). Se obtuvieron mediante la implementacion de un web scraper que descarga los perfiles públicos de los diputados federales de la Cámara de Diputados del H. Congreso de la Unión, abarcando las legislaturas LVII a LXVI (1997–presente), con aproximadamente 500 diputados por legislatura.

Cada corrida del scraper produce un archivo CSV por legislatura. Cada archivo contiene una fila por perfil descargado y 32 columnas que recogen tanto los datos personales planos como las secciones de trayectoria y comisiones, serializadas como listas JSON dentro de una celda.

La unidad de análisis es el legislador dentro de una legislatura. Un mismo diputado puede aparecer en más de un archivo si fue reelecto en legislaturas distintas. El campo `diputado_id` permite vincular registros del mismo individuo entre legislaturas (se calcula como SHA-256[:12] sobre nombre normalizado y fecha de nacimiento).

---

### Columnas del CSV crudo (extracto del web scraper)

#### Identificadores

| Campo | Tipo | Descripción |
|---|---|---|
| **diputado_id** | str | ID estable entre legislaturas: SHA-256[:12] calculado sobre nombre normalizado + fecha de nacimiento. Vacío si faltan nombre o nacimiento. |
| **referencia** | int | ID numérico del perfil en el SIL. Identifica de forma única a un legislador dentro de una legislatura, pero puede cambiar entre legislaturas. |
| **legislatura_num** | int | Número entero de la legislatura (57–66). Asignado por el scraper según la legislatura objetivo. |
| **profile_url** | str | URL directa al perfil del legislador en el SIL. Útil para diagnóstico y verificación manual. |

#### Datos personales (tabla principal del perfil)

| Campo | Tipo | Descripción |
|---|---|---|
| **nombre** | str | Nombre completo del legislador extraído del encabezado del perfil. Los prefijos de cargo (Diputado/a, Sen., Lic., etc.) son eliminados automáticamente. |
| **numero_de_la_legislatura** | str | Número de la legislatura tal como aparece en el perfil (p. ej. "LXVI Legislatura"). |
| **periodo_de_la_legislatura** | str | Años del período legislativo (p. ej. "2024-2027"). |
| **partido** | str | Nombre o siglas del partido político al que pertenece el legislador según el SIL. |
| **nacimiento** | str | Fecha de nacimiento en formato DD/MM/YYYY. Puede estar vacía en perfiles incompletos. |
| **entidad** | str | Entidad federativa que representa o por la que fue electo. |
| **ciudad** | str | Ciudad de nacimiento o de referencia del legislador, según disponibilidad en el perfil. |
| **principio_de_eleccion** | str | Tipo de elección: mayoría relativa (MR) o representación proporcional (RP). |
| **ubicacion** | str | Distrito o circunscripción electoral. |
| **correo_electronico** | str | Correo institucional publicado en el perfil. Frecuentemente vacío o desactualizado. |
| **telefono** | str | Teléfono institucional publicado en el perfil. Frecuentemente vacío. |
| **suplente** | str | Nombre completo del suplente registrado. Vacío si no tiene suplente. |
| **suplente_referencia** | str | ID del perfil del suplente en el SIL. Vacío si no tiene suplente. |
| **ultimo_grado_de_estudios** | str | Nivel de escolaridad declarado (p. ej. "Licenciatura", "Maestría", "Doctorado"). Texto libre; varía en ortografía entre perfiles. |
| **preparacion_academica** | str | Campo de formación o carrera declarada. Texto libre (p. ej. "Derecho", "Ingeniería Civil"). |
| **experiencia_legislativa** | str | Texto libre que describe cargos legislativos previos. Puede incluir referencias a diputaciones locales, federales o senadurías anteriores. |
| **redes_sociales** | str | Cuentas de redes sociales publicadas en el perfil. Frecuentemente vacías o desactualizadas. |

#### Secciones anidadas (serializadas como JSON)

Cada una de estas columnas contiene una lista de diccionarios serializada como cadena JSON. Los registros vacíos se guardan como `"[]"`. La mayoría de las secciones de trayectoria tienen la estructura `[{"Del año": "...", "Al año": "...", "Experiencia": "..."}, ...]`.

| Campo | Estructura JSON | Descripción |
|---|---|---|
| **comisiones** | `[{Comisión, Puesto, Fecha Inicial, Fecha Final, Estatus}]` | Membresías en comisiones legislativas (ordinarias, especiales, de investigación). Incluye el puesto (Integrante, Secretario, Presidente, etc.) y estatus (activo/concluido). |
| **licencias_reincorporaciones** | `[{Del año, Al año, Experiencia}]` | Registro de licencias tomadas y reincorporaciones al cargo durante la legislatura. |
| **trayectoria_administrativa** | `[{Del año, Al año, Experiencia}]` | Cargos en la administración pública (federal, estatal o municipal), organizaciones civiles, sindicatos, partidos, etc. |
| **trayectoria_legislativa** | `[{Del año, Al año, Experiencia}]` | Cargos legislativos previos (diputaciones locales o federales anteriores, senadurías). |
| **trayectoria_politica** | `[{Del año, Al año, Experiencia}]` | Participación en partidos políticos, candidaturas, militancia y cargos internos partidistas. |
| **trayectoria_academica** | `[{Del año, Al año, Experiencia}]` | Estudios realizados: institución, grado y período. Puede incluir cursos, diplomados y formación en el extranjero. |
| **trayectoria_empresarial** | `[{Del año, Al año, Experiencia}]` | Actividad en el sector privado o iniciativa privada: cargos directivos, empresas propias, consultoría. |
| **otros_rubros** | `[{Del año, Al año, Experiencia}]` | Actividades no clasificadas en las demás categorías. En la legislatura LXVI contiene además la subsección de investigación y docencia antes de que el raspador la separe. |
| **organos_de_gobierno** | `[{...}]` | Participación en órganos de gobierno de organismos autónomos, fideicomisos u otras instituciones. Estructura variable según el perfil. |
| **observaciones** | `[{...}]` | Notas adicionales del perfil. Estructura variable; frecuentemente vacío. |

#### Diagnóstico

| Campo | Tipo | Descripción |
|---|---|---|
| **error** | str | Vacío si el perfil fue descargado y procesado sin incidencias. Contiene el mensaje de error (`fetch_failed`, etc.) si la descarga falló. Los perfiles con error tienen el resto de sus campos vacíos. |

---

### Parametros conocidos de datos

**Conocido**: fuente de datos (SIL, público), cobertura temporal (LVII–LXVI, 1997–presente), unidad de análisis (legislador × legislatura), estructura de cada columna, mecanismo de reanudación automática ante interrupciones, método de generación del `diputado_id`.

**Desconocido / evidencia insuficiente**: tasa de perfiles con error por legislatura; completitud real de los campos de texto libre (`preparacion_academica`, `experiencia_legislativa`) dado que son declarativos y su llenado depende de cada legislador; posibles duplicados dentro de una misma legislatura si un legislador aparece en más de un partido (el raspador deduplica por referencia, pero el SIL podría asignar referencias distintas al mismo individuo); cobertura real de `licencias_reincorporaciones` en legislaturas anteriores a LXII; grado de estandarización de los campos de texto libre entre legislaturas.

### Tratamiento de datos diputrax

#### Datos explotados para modelo (Diccionario de Datos — Diputados Federales)

Los datos extraidos mediante el web scraper son tratados con el fin de generar una fuenta de datos integrada y optimizada para el analisis de modelos de machine learning. Como resultado de la segunda capa de tratamiento de datos el modelo cuenta con 78 variables registradas por diputado, organizadas en ocho grupos temáticos según el diccionario siguiente.

La unidad de análisis es el diputado en una legislatura determinada. El tratamiento de segundo nivel no garantiza unicidad: un mismo individuo puede aparecer en múltiples legislaturas con un registro distinto por periodo. Se asume que cada entrada corresponde a la participación de un diputado en una legislatura específica.

Dentro del proyecto diputrax se configuro un pipeline (`run_pipeline.py`) que consolida diez archivos CSV de origen (uno por legislatura) en un único archivo Parquet limpio ubicado en `data/database/clean/`, y genera sobre él un reporte de análisis exploratorio en `reports/eda/`. El pipeline añade además dos variables derivadas: `partido_mayoria` y `es_partido_mayoria`.

---

### Grupo 1 — Identificadores

| Campo | Descripción |
|---|---|
| **diputado_id** | Hash único del diputado. |
| **referencia** | ID numérico en la fuente original. |
| **nombre** | Nombre completo del diputado. |
| **legislatura_nombre** | Nombre romano de la legislatura (ej. LIX). |
| **legislatura_num** | Número ordinal de la legislatura (57–66). |
| **partido_nombre** | Nombre completo del partido político. |
| **partido** | Siglas del partido. |
| **partido_mayoria** | Partido con pluralidad de escaños en esa legislatura *(derivada por el pipeline)*. |
| **es_partido_mayoria** | Binario: 1 si el diputado pertenece al partido mayoritario de su legislatura *(variable objetivo principal)*. |
| **source_file** | Archivo CSV de origen. |

---

### Grupo 2 — Datos Personales

| Campo | Descripción |
|---|---|
| **y_nacimiento** | Año de nacimiento. |
| **edad_al_tomar_cargo** | Edad al inicio del mandato legislativo. |
| **grado_estudios_ord** | Nivel educativo ordinal (0 = sin estudios formales, 7 = doctorado). |
| **area_formacion** | Área disciplinaria de estudios (ej. Derecho, Economía, Medicina). |
| **en_licencia** | `True` si estuvo en licencia durante el periodo. |
| **suplente_referencia** | ID de referencia del diputado suplente. |
| **tiene_suplente** | 1 si tiene suplente registrado. |
| **mayoria_relativa** | 1 si fue electo por mayoría relativa; 0 si por representación proporcional. |
| **entidad_codigo** | Código de 3 letras del estado (ej. CDMX, JAL, NL). |
| **distrito_circ** | Número y nombre del distrito uninominal o circunscripción plurinominal. |

---

### Grupo 3 — Comisiones

| Campo | Descripción |
|---|---|
| **n_comisiones** | Total de comisiones ordinarias a las que perteneció. |
| **n_comisiones_especiales** | Total de comisiones especiales. |
| **n_presidencias** | Número de presidencias de comisión ejercidas. |
| **n_secretarias** | Número de secretarías de comisión ejercidas. |
| **presidente_comision** | 1 si presidió al menos una comisión. |
| **lider_comision** | 1 si fue presidente o secretario de alguna comisión. |
| **n_comisiones_nodales** | Comisiones de alta influencia legislativa (ej. Presupuesto, Hacienda). |
| **n_comisiones_tematicas** | Comisiones de política sectorial (ej. Salud, Educación). |
| **n_comisiones_lastre** | Comisiones de bajo perfil político. |

---

### Grupo 4 — Trayectoria Legislativa

| Campo | Descripción |
|---|---|
| **fue_diputado_local** | 1 si fue diputado local antes del cargo federal. |
| **fue_diputado_federal** | 1 si fue diputado federal en alguna legislatura previa. |
| **fue_senador** | 1 si fue senador de la República. |
| **n_cargos_legislativos_prev** | Suma de cargos legislativos previos (diputado local + federal + senador). |
| **n_trayectoria_legislativa** | Conteo ponderado de experiencia legislativa acumulada. |

---

### Grupo 5 — Trayectoria Administrativa

| Campo | Descripción |
|---|---|
| **n_trayectoria_admin** | Conteo total de cargos administrativos desempeñados. |
| **fue_presidente_mun** | 1 si fue presidente municipal. |
| **fue_presidente_org** | 1 si presidió algún organismo público o privado. |
| **fue_director_general** | 1 si fue director general. |
| **fue_secretario_cargo** | 1 si fue secretario de despacho (federal o estatal). |
| **fue_subsecretario** | 1 si fue subsecretario. |
| **fue_director** | 1 si fue director de área. |
| **fue_coordinador** | 1 si fue coordinador. |
| **fue_delegado** | 1 si fue delegado. |
| **fue_asesor** | 1 si fue asesor. |
| **fue_regidor** | 1 si fue regidor en cabildo municipal. |
| **fue_sindico** | 1 si fue síndico municipal. |
| **admin_en_partido** | 1 si tuvo cargo administrativo dentro de un partido político. |
| **admin_en_sindicato** | 1 si tuvo cargo en un sindicato. |
| **admin_en_universidad** | 1 si tuvo cargo en una institución universitaria. |
| **admin_en_gobierno_fed** | 1 si tuvo cargo en el gobierno federal. |
| **admin_en_gobierno_est** | 1 si tuvo cargo en un gobierno estatal. |
| **admin_en_gobierno_mun** | 1 si tuvo cargo en un gobierno municipal. |
| **nivel_cargo_max** | Nivel máximo de cargo administrativo alcanzado (0 = ninguno, 5 = secretario de estado). |

---

### Grupo 6 — Trayectoria Juvenil

| Campo | Descripción |
|---|---|
| **tiene_exp_juvenil** | 1 si tiene experiencia documentada en organizaciones juveniles. |
| **lider_juvenil_partido** | 1 si fue líder juvenil de partido. |
| **lider_juvenil_gobierno** | 1 si tuvo cargo juvenil en gobierno. |
| **miembro_org_juvenil** | 1 si fue miembro de una organización juvenil. |
| **nivel_liderazgo_juvenil** | Nivel de liderazgo juvenil (0–3). |

---

### Grupo 7 — Formación Académica

| Campo | Descripción |
|---|---|
| **tiene_posgrado** | 1 si tiene estudios de posgrado (maestría o superior). |
| **tiene_doctorado** | 1 si tiene doctorado. |
| **estudios_en_extranjero** | 1 si realizó estudios en el extranjero. |
| **univ_publica** | 1 si estudió en universidad pública. |
| **univ_privada** | 1 si estudió en universidad privada. |
| **univ_extranjera** | 1 si estudió en universidad extranjera. |
| **acad_unam** | 1 si estudió en la UNAM. |
| **acad_itesm** | 1 si estudió en el ITESM (Tec de Monterrey). |
| **acad_itam** | 1 si estudió en el ITAM. |
| **acad_ibero** | 1 si estudió en la Universidad Iberoamericana. |
| **acad_udg** | 1 si estudió en la Universidad de Guadalajara. |
| **acad_ipn** | 1 si estudió en el IPN. |
| **acad_uam** | 1 si estudió en la UAM. |
| **acad_anahuac** | 1 si estudió en la Universidad Anáhuac. |
| **acad_uanl** | 1 si estudió en la UANL. |
| **acad_uv** | 1 si estudió en la Universidad Veracruzana. |

---

### Grupo 8 — Conteos de Trayectoria

| Campo | Descripción |
|---|---|
| **n_trayectoria_politica** | Conteo general de cargos políticos (partido, gobierno, legislativo). |
| **n_trayectoria_empresarial** | Conteo de cargos en el sector privado o empresarial. |
| **n_investigacion_docencia** | Conteo de actividades académicas o de investigación. |
| **n_organos_gobierno** | Conteo de participaciones en órganos de gobierno colegiados. |

---

**Conocidos e incógnitas**:

**Conocido**: definición de la variable objetivo, lista de características, legislaturas cubiertas (LVII–LXVI), fuente institucional (CAMARA), lógica de construcción del dataset.

**Desconocido / evidencia insuficiente**: criterio de inclusión de registros (si todos los 500 diputados por legislatura tienen un registro exhaustivo o si para alguna categorias existe sesgo de selección), tasa de cobertura real de variables biográficas, posible doble conteo de diputados que reingresan en múltiples legislaturas, homogeneidad del proceso de codificación a lo largo del tiempo, y representatividad temporal ante cambios institucionales (reforma política 2014, nueva correlación de fuerzas 2018).



In [ ]:
# ============================================================
# CÁLCULO DE PORCENTAJE DE NULOS POR ÉPOCA
# ============================================================
# Este bloque calcula, para cada época legislativa, qué porcentaje
# de valores faltantes (NaN) tiene la variable "y_nacimiento".

missing_by_era = (
    # Agrupa el DataFrame "plot_df" por la columna "era_label".
    # Cada grupo representa una época.
    plot_df.groupby("era_label")[["y_nacimiento"]]

    # Para cada grupo, calcula el porcentaje de valores nulos.
    # g.isna() devuelve True/False según si cada valor es nulo.
    # mean() sobre booleanos calcula la proporción de True.
    # Multiplicar por 100 convierte esa proporción en porcentaje.
    .apply(lambda g: g.isna().mean() * 100)

    # Renombra explícitamente el nombre del índice como "era_label".
    # Esto ayuda a que la estructura sea más clara al resetear índice.
    .rename_axis(index="era_label")

    # Convierte el índice nuevamente en una columna normal.
    # El resultado final es un DataFrame plano.
    .reset_index()
)

# Muestra la tabla resultante redondeando a 2 decimales.
# Sirve para inspeccionar numéricamente el porcentaje de nulos por época.
display(missing_by_era.round(2))


# ============================================================
# PREPARACIÓN DE DATOS PARA EL HEATMAP
# ============================================================
# Convierte la columna "era_label" en índice para que aparezca
# como eje Y del mapa de calor.
heat = missing_by_era.set_index("era_label")

# Crea la figura y el eje donde se dibujará el heatmap.
# figsize=(10, 4) define un gráfico ancho y relativamente bajo.
fig, ax = plt.subplots(figsize=(10, 4))

# Dibuja un mapa de calor con seaborn:
# - heat: tabla de datos a visualizar
# - annot=True: escribe el valor numérico dentro de cada celda
# - fmt=".2f": muestra los números con 2 decimales
# - cmap="YlOrRd": paleta de colores amarillo-naranja-rojo
# - cbar_kws: personaliza la barra lateral de color
# - ax=ax: dibuja sobre el eje creado antes
sns.heatmap(
    heat,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    cbar_kws={"label": "% nulos"},
    ax=ax
)

# Título principal del gráfico.
ax.set_title("Porcentaje de nulos por época")

# Etiqueta del eje X.
ax.set_xlabel("Variable")

# Etiqueta del eje Y.
ax.set_ylabel("Época")

# Rota las etiquetas del eje X en 90 grados para que no se encimen.
ax.tick_params(axis="x", labelrotation=90)

# Ajusta automáticamente los espacios del gráfico para evitar recortes.
plt.tight_layout()

# Muestra el gráfico final en pantalla.
plt.show()

# 3. Cumplimiento normativo y  requerimientos de interpretabilidad del modelo

sdfasdfasdfsdf

# 4. Alcance del analisis

# 5. Fuera del alcance del analisis

# 6. Analisis exploratorio de datos (EDA)

dfdfdfdf